In [1]:
# ========================
# Turn的高峰/特徵與GPT動作+說話對應分析
# 提煉可直接用於查表的規則雛形
# ========================
import pandas as pd
import json
import numpy as np
from pathlib import Path

In [2]:
# 讀取資料
folder = Path("C:/Users/AW'z/Downloads/ballet_Analysis_Results/260201__analysis_metrics/26020113_analysis_metrics/")
json_file = folder / "ceremony_dialogue_2026020113.json"
energy_file = folder / "energy.csv"
geometry_file = folder / "geometry.csv"
stability_file = folder / "stability.csv"
sync_file = folder / "synchronization.csv"
trans_file = folder / "transition.csv"
skeleton_file = folder / "26020113_skeleton.csv"

In [3]:
# 讀取GPT對話
with open(json_file, 'r', encoding='utf-8') as f:
    gpt_data = json.load(f)

# 讀取指標資料
energy_df = pd.read_csv(energy_file)
geometry_df = pd.read_csv(geometry_file)
stability_df = pd.read_csv(stability_file)
sync_df = pd.read_csv(sync_file)
trans_df = pd.read_csv(trans_file)
skeleton_df = pd.read_csv(skeleton_file)

# 為每個turn建立詳細分析
turn_analysis = []

for turn_data in gpt_data.get("dialogue", []):
    turn_num = turn_data["turn"]
    frame = turn_data["frame"]
    timestamp = turn_data["timestamp_sec"]
    
    # 提取AI sees和AI says
    ai_response = turn_data["ai_response"]
    ai_sees = ""
    ai_says = ""
    
    if "【AI sees】" in ai_response:
        sees_part = ai_response.split("【AI sees】")[-1]
        if "【AI says】" in sees_part:
            ai_sees = sees_part.split("【AI says】")[0].strip()
            ai_says = ai_response.split("【AI says】")[-1].strip()
        else:
            ai_sees = sees_part.strip()
    
    # 獲取該frame的指標數據（取前後5幀的平均值以平滑）
    frame_range = range(max(0, frame-5), min(len(energy_df), frame+6))
    
    energy_avg = energy_df.iloc[frame_range]['energy'].mean() if frame < len(energy_df) else 0
    energy_peak = energy_df.iloc[frame_range]['energy'].max() if frame < len(energy_df) else 0
    
    # 幾何特徵
    if frame < len(geometry_df):
        volume_avg = geometry_df.iloc[frame_range]['volume'].mean()
        curvature_avg = geometry_df.iloc[frame_range]['curvature'].mean()
    else:
        volume_avg = curvature_avg = 0
    
    # 穩定性特徵
    if frame < len(stability_df):
        sway_avg = stability_df.iloc[frame_range]['sway'].mean()
    else:
        sway_avg = 0
    
    # 同步性特徵
    if frame < len(sync_df):
        left_avg = sync_df.iloc[frame_range]['left'].mean()
        right_avg = sync_df.iloc[frame_range]['right'].mean()
        correlation_avg = sync_df.iloc[frame_range]['correlation'].mean()
    else:
        left_avg = right_avg = correlation_avg = 0
    
    # 轉換特徵 (扭力和急動度)
    if frame < len(trans_df):
        torque_avg = trans_df.iloc[frame_range]['torque'].mean()
        jerk_avg = trans_df.iloc[frame_range]['jerk'].mean()
    else:
        torque_avg = jerk_avg = 0
    
    # 提取動作關鍵詞
    ballet_moves = []
    move_keywords = ['pirouette', 'arabesque', 'fouetté', 'jeté', 'plié']
    for move in move_keywords:
        if move in ai_sees.lower():
            ballet_moves.append(move)
    
    # 提取文化參照
    cultural_refs = []
    if '睡美人' in ai_sees or '奧蘿拉' in ai_sees:
        cultural_refs.append('睡美人/奧蘿拉')
    if '天鵝湖' in ai_sees or '奧黛特' in ai_sees:
        cultural_refs.append('天鵝湖/奧黛特')
    if '胡桃夾子' in ai_sees or '糖梅仙子' in ai_sees:
        cultural_refs.append('胡桃夾子/糖梅仙子')
    
    # 提取情感/意象關鍵詞
    emotion_keywords = []
    emotion_terms = ['優雅', '輕盈', '力量', '控制', '掙扎', '純潔', '夢幻', '謙遜']
    for term in emotion_terms:
        if term in ai_sees:
            emotion_keywords.append(term)
    
    turn_analysis.append({
        'turn': turn_num,
        'timestamp': timestamp,
        'frame': frame,
        'energy_avg': round(energy_avg, 2),
        'energy_peak': round(energy_peak, 2),
        'volume': round(volume_avg, 4),
        'curvature': round(curvature_avg, 2),
        'sway': round(sway_avg, 3),
        'left_sync': round(left_avg, 3),
        'right_sync': round(right_avg, 3),
        'correlation': round(correlation_avg, 3),
        'torque': round(torque_avg, 3),
        'jerk': round(jerk_avg, 2),
        'ballet_moves': ', '.join(ballet_moves) if ballet_moves else 'N/A',
        'cultural_refs': ', '.join(cultural_refs) if cultural_refs else 'N/A',
        'emotion_keywords': ', '.join(emotion_keywords) if emotion_keywords else 'N/A',
        'ai_sees_full': ai_sees,
        'ai_says_full': ai_says
    })

# 轉換為DataFrame
analysis_df = pd.DataFrame(turn_analysis)

In [4]:
# ========================
# 規則雛形提煉
# ========================
print("=" * 80)
print("Turn的高峰/特徵與GPT動作+說話對應分析")
print("=" * 80)
print()

# 顯示每個turn的詳細分析
for idx, row in analysis_df.iterrows():
    print(f"\n{'='*60}")
    print(f"Turn {row['turn']} (時間: {row['timestamp']}秒, Frame: {row['frame']})")
    print(f"{'='*60}")
    print(f"\n【指標特徵】")
    print(f"  能量平均: {row['energy_avg']:.2f} | 能量峰值: {row['energy_peak']:.2f}")
    print(f"  體積: {row['volume']:.4f} | 曲率: {row['curvature']:.2f}")
    print(f"  搖擺: {row['sway']:.3f}")
    print(f"  左肢同步: {row['left_sync']:.3f} | 右肢同步: {row['right_sync']:.3f} | 相關性: {row['correlation']:.3f}")
    print(f"  扭力: {row['torque']:.3f} | 急動度: {row['jerk']:.2f}")
    print(f"\n【識別動作】")
    print(f"  芭蕾動作: {row['ballet_moves']}")
    
    print(f"\n【文化參照】")
    print(f"  {row['cultural_refs']}")
    
    print(f"\n【情感意象】")
    print(f"  {row['emotion_keywords']}")
    
    print(f"\n【AI視覺描述】")
    print(f"  {row['ai_sees_full'][:200]}...")
    
    print(f"\n【AI說話內容】")
    print(f"  {row['ai_says_full'][:200]}..." if row['ai_says_full'] else "  (無)")

Turn的高峰/特徵與GPT動作+說話對應分析


Turn 1 (時間: 0.0秒, Frame: 0)

【指標特徵】
  能量平均: 0.40 | 能量峰值: 0.57
  體積: 0.0479 | 曲率: 5.84
  搖擺: 0.080
  左肢同步: 1.923 | 右肢同步: 2.347 | 相關性: 0.000
  扭力: 3.082 | 急動度: 22658.37

【識別動作】
  芭蕾動作: arabesque, jeté

【文化參照】
  天鵝湖/奧黛特

【情感意象】
  優雅

【AI視覺描述】
  如同在《天鵝湖》中的水面上，舞者以優雅的arabesque劃出一道柔美的弧線，隨後又以強勁的jeté跳躍，彷彿空氣中迸發著激情的流星。...

【AI說話內容】
  親愛的舞者，您是否感受到那股純粹的光芒在每一次旋轉和飛翔之間流淌？願您的舞步如星辰般閃耀，無懼於舞台上的一切。...

Turn 2 (時間: 2.0秒, Frame: 60)

【指標特徵】
  能量平均: 1.72 | 能量峰值: 2.25
  體積: 0.0890 | 曲率: 6.24
  搖擺: 0.040
  左肢同步: 4.125 | 右肢同步: 4.918 | 相關性: 0.675
  扭力: 2.120 | 急動度: 3046.41

【識別動作】
  芭蕾動作: pirouette, fouetté

【文化參照】
  N/A

【情感意象】
  優雅

【AI視覺描述】
  在您的舞姿中，像Odile般的旋轉與引誘交錯，宛如冷冽的黑天鵝，流暢的fouettés與優雅的pirouette交織出一幅深邃而華麗的畫面。...

【AI說話內容】
  親愛的舞者，您是否感受到那份黑色的魅力在每一次轉身中回響？願您的舞步如珠寶般璀璨，無畏於誘惑的深淵。...

Turn 3 (時間: 4.0秒, Frame: 120)

【指標特徵】
  能量平均: 0.73 | 能量峰值: 1.50
  體積: 0.0424 | 曲率: 9.20
  搖擺: 0.072
  左肢同步: 3.252 | 右肢同步: 2.940 | 相關性: 0.591
  扭力: 3.464 | 急動度: 6960.45

【識別動作】
  芭蕾動作: arabesque, plié

【文化參照】

In [5]:
# ========================
# 查表規則雛形
# ========================

print("\n\n" + "=" * 80)
print("查表規則雛形 (Rule Prototypes for Lookup Table)")
print("=" * 80)

# 規則1: 動作類型對應
print("\n【規則1: 動作類型對應】")
print("-" * 60)
move_mapping = analysis_df.groupby('ballet_moves').agg({
    'energy_avg': 'mean',
    'sway': 'mean',
    'cultural_refs': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'N/A'
}).round(3)
print(move_mapping)

# 規則2: 能量區間對應文化參照
print("\n【規則2: 能量區間對應文化參照】")
print("-" * 60)
analysis_df['energy_level'] = pd.cut(analysis_df['energy_avg'], 
                                      bins=[0, 5, 10, 20, 300],
                                      labels=['低', '中', '高', '極高'])
energy_culture = analysis_df.groupby('energy_level')['cultural_refs'].apply(
    lambda x: x.value_counts().head(3).to_dict()
)
for level, refs in energy_culture.items():
    print(f"  能量{level}: {refs}")

# 規則3: 體積與動作類型關聯
print("\n【規則3: 體積與動作類型關聯】")
print("-" * 60)
volume_moves = analysis_df.groupby('ballet_moves')['volume'].agg(['mean', 'min', 'max']).round(4)
print(volume_moves)

# 規則4: 情感關鍵詞頻率統計
print("\n【規則4: 情感關鍵詞頻率統計】")
print("-" * 60)
all_emotions = []
for emotions in analysis_df['emotion_keywords']:
    if emotions != 'N/A':
        all_emotions.extend(emotions.split(', '))
emotion_freq = pd.Series(all_emotions).value_counts()
print(emotion_freq)

# 規則5: 綜合特徵模式
print("\n【規則5: 綜合特徵模式 - 可直接用於查表】")
print("-" * 60)
print("格式: [動作類型] + [能量範圍] + [體積範圍] → [文化參照] + [情感關鍵詞]")
print()

for idx, row in analysis_df.iterrows():
    if row['ballet_moves'] != 'N/A':
        rule = f"IF 動作={row['ballet_moves']} AND 能量={row['energy_level']} AND 體積≈{row['volume']:.4f}"
        result = f"THEN 參照={row['cultural_refs']}, 情感={row['emotion_keywords']}"
        print(f"Rule {idx+1}: {rule}")
        print(f"         {result}")
        print()



查表規則雛形 (Rule Prototypes for Lookup Table)

【規則1: 動作類型對應】
------------------------------------------------------------
                    energy_avg   sway cultural_refs
ballet_moves                                       
arabesque, jeté           0.58  0.264           N/A
arabesque, plié           0.73  0.072           N/A
jeté, plié                2.58  0.271           N/A
pirouette, fouetté        1.72  0.040           N/A

【規則2: 能量區間對應文化參照】
------------------------------------------------------------
  能量('低', 'N/A'): 4.0
  能量('低', '天鵝湖/奧黛特'): 1.0
  能量('中', 'N/A'): nan
  能量('中', '天鵝湖/奧黛特'): nan
  能量('高', 'N/A'): nan
  能量('高', '天鵝湖/奧黛特'): nan
  能量('極高', 'N/A'): nan
  能量('極高', '天鵝湖/奧黛特'): nan

【規則3: 體積與動作類型關聯】
------------------------------------------------------------
                      mean     min     max
ballet_moves                              
arabesque, jeté     0.0957  0.0479  0.1435
arabesque, plié     0.0424  0.0424  0.0424
jeté, plié          0.0774  0.0774  0.0774


C:\Users\AW'z\AppData\Local\Temp\ipykernel_21620\3315887638.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  energy_culture = analysis_df.groupby('energy_level')['cultural_refs'].apply(


In [6]:
# 儲存分析結果
output_file = folder.parent / "turn_analysis_mapping.csv"
analysis_df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n分析結果已儲存至: {output_file}")

# 儲存規則雛形為JSON
rules = {
    'move_energy_mapping': move_mapping.to_dict(),
    'energy_culture_mapping': {str(k): v for k, v in energy_culture.items()},
    'volume_move_mapping': volume_moves.to_dict(),
    'emotion_frequency': emotion_freq.to_dict(),
    'detailed_rules': analysis_df[['turn', 'ballet_moves', 'energy_level', 'volume', 
                                    'cultural_refs', 'emotion_keywords']].to_dict('records')
}

rules_file = folder.parent / "lookup_rules.json"
with open(rules_file, 'w', encoding='utf-8') as f:
    json.dump(rules, f, ensure_ascii=False, indent=2)
print(f"規則雛形已儲存至: {rules_file}")


分析結果已儲存至: C:\Users\AW'z\Downloads\ballet_Analysis_Results\260201__analysis_metrics\turn_analysis_mapping_13.csv
規則雛形已儲存至: C:\Users\AW'z\Downloads\ballet_Analysis_Results\260201__analysis_metrics\lookup_rules_13.json
